# OCS — Resting-state main analysis (XCP-D 0.6.0, fsLR 91k)

QC and analysis of the OCS (marsonc) resting-state data processed with XCP-D 0.6.0 (fMRIPrep 23.2.0).

**Run on cortex** from the `ocs` conda env (`./setup_env.sh`). Paths come from `ocs.paths`
(override with `OCS_XCPD_DIR` / `OCS_FMRIPREP_DIR`). Working tables go to `notebooks/dfs/`
and figures to `notebooks/figures/`; both are git-ignored, so no participant data is committed.
NM-MRI masks are not in the public repo: copy them to `resources/private/akili_NM_masks/` (or set `OCS_NM_MASKS`).

**Sections**
1. Inventory of all run files
2. Header checks: TR, run length, which FD threshold XCP-D applied
3. Motion QC from the DCAN HDF5 files (+ `motion.tsv`) and run/session inclusion
4. ROI QC for selected seed ROIs (coverage, signal, homogeneity, global-signal coupling, tSNR)
5. Final analysis sample
6. Analysis input per session: combined runs (default) or the best run by motion (`SESSION_INPUT`, CELL 1)

### Notes:
- runs --> one row per sub/ses/run with paths to dtseries (denoised, smoothed), DCAN hdf5, motion.tsv, atlas timeseries, fMRIPrep dtseries; timepoint = session order; dfs/runs.csv (CELL 3)
- headers --> runs + TR, frames, grayordinates, uncensored length, matching FD thresholds, header flags; dfs/headers.csv (CELL 4)
- motion_long --> every run x every DCAN FD threshold (frames/seconds remaining, mean FD); dfs/motion_long.csv (CELL 5)
- motion_qc --> one row per run at the XCP-D FD threshold + uncensored FD stats; adds ok_seconds, ok_mean_fd, run_ok; dfs/motion_qc.csv (CELL 5)
- session_qc --> per sub/ses: runs ok, seconds ok, session_ok, paired_ok (motion only); dfs/session_qc.csv (CELL 6)
- roi_index --> every ROI grayordinate (roi, hemi L/R/M, MNI xyz, CIFTI structure) for the NIfTI ROIs: Murty_VTA_p50/p75, CIT168_SNcPBPVTA, NM_VTA (partial-volume sampled, min_coverage 1/8 -> 9 grayordinates), NM_SN (215), NM_SNc (32); dfs/roi_index.csv (CELL 8)
- roi_qc --> per run x ROI x hemi: coverage, sd_ratio, homogeneity, pc1_var, r_global, tSNR; atlas ROIs: columns found / NaN coverage; dfs/roi_qc.csv (CELL 9)
- roi_agree --> per run: correlation between bilateral mean time series of each ROI-definition pair; dfs/roi_agree.csv (CELL 9)
- run_qc --> motion_qc + FOCUS_ROI flags (coverage, robust-z outliers), roi_ok, usable = run_ok & roi_ok; dfs/run_qc.csv (CELL 10)
- final_sample --> per sub/ses from usable runs: usable seconds, runs_usable, session_usable, paired_usable; dfs/final_sample.csv (CELL 12)
- analysis_inputs --> final_sample collapsed to one input dtseries per sub/ses (SESSION_INPUT: combined file if built from exactly the usable runs, else best usable run by BEST_RUN_RANK); input, runs_used, reason, dtseries/dtseries_smoothed, seconds, mean_fd_remaining, input_ok, paired_ok; dfs/analysis_inputs.csv (CELL 13)

## Setup & parameters

In [ ]:
# ==========================================
# CELL 1: SETUP & PARAMETERS
# ==========================================
# Imports, data/output paths and all QC parameters. Edit the PARAMETERS block only.
# ROI definitions: volumetric NIfTI masks in MNI (NLin6Asym) applied to the CIFTI
# subcortical voxels, plus XCP-D atlas parcels read from *_timeseries.tsv for comparison.

import os, warnings
from pathlib import Path
import numpy as np, pandas as pd, nibabel as nib
import matplotlib.pyplot as plt
import ocs
from ocs import XCPD_DIR, FMRIPREP_DIR, RESOURCES

NBDIR = ocs.REPO_DIR / 'notebooks'
DFS, FIG = NBDIR / 'dfs', NBDIR / 'figures'
DFS.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)

# ---------------- PARAMETERS ----------------
TASK               = 'rest'
EXPECTED_TR        = 0.8      # s; fMRIPrep TR was corrected to 0.8 (fix_fmriprep_TR_0.8.log)
EXPECTED_GRAYORDS  = 91282
FD_REPORT          = [0.2, 0.3, 0.5]   # thresholds summarised from the DCAN files
MIN_RUN_SECONDS    = 120      # run kept if >= this many seconds survive censoring  (CONFIRM)
MIN_SESSION_SECONDS= 300      # session kept if kept runs total >= this             (CONFIRM)
MAX_MEAN_FD        = 0.30     # mm, mean FD of remaining frames; runs above are flagged (CONFIRM)
ATLASES            = ['4S356Parcels', 'Tian']

CIT = RESOURCES / 'atlases/CIT168_Pauli2018'; MUR = RESOURCES / 'atlases/Murty2014'
# NM-MRI template masks (1 mm MNI; shared on request - kept in git-ignored resources/private, copy to cortex by hand)
NM = Path(os.environ.get('OCS_NM_MASKS', RESOURCES / 'private/akili_NM_masks'))
ROIS = {   # name: dict(kind='nifti', path, threshold | labels [, min_coverage])  or  dict(kind='atlas', atlas, columns)
    'Murty_VTA_p50':      dict(kind='nifti', path=MUR / 'Murty2014_VTA_bilateral_prob_MNI152_res-2mm.nii.gz', threshold=0.50),
    'Murty_VTA_p75':      dict(kind='nifti', path=MUR / 'Murty2014_VTA_bilateral_prob_MNI152_res-2mm.nii.gz', threshold=0.75),
    'CIT168_SNcPBPVTA':   dict(kind='nifti', path=CIT / 'CIT168_det_MNI152NLin2009cAsym_res-0.7mm.nii.gz', labels=[7, 10, 11]),
    # 29 x 1 mm voxels: centre sampling keeps 3 grayordinates; min_coverage=1/8 keeps every 2 mm voxel the mask touches
    'NM_VTA':             dict(kind='nifti', path=NM / 'VTA_mask_50.nii.gz', threshold=0.5, min_coverage=0.125),
    'NM_SN':              dict(kind='nifti', path=NM / 'SN_mask.nii.gz', threshold=0.5),        # 1,807 x 1 mm -> 215 grayordinates
    'NM_SNc':             dict(kind='nifti', path=NM / 'SNc_mask_50.nii.gz', threshold=0.5),    # 249 x 1 mm -> 32 grayordinates
    'XCPD_4S_SNcPBPVTA':  dict(kind='atlas', atlas='4S356Parcels', columns=['LH-SNc_PBP_VTA', 'RH-SNc_PBP_VTA']),
}
FOCUS_ROI = 'Murty_VTA_p50'   # ROI shown in the detailed ROI-QC figures

# Analysis input per session (CELL 13)
SESSION_INPUT = 'combined'    # 'combined' (default): run-concatenated session file (scripts/combine_runs.py)
                              # 'best_run' : the single usable run with the best motion metrics
BEST_RUN_RANK = [('fraction_remaining', False),   # (motion_qc column, ascending?) - 1st = primary:
                 ('mean_fd_remaining', True)]     # most frames retained, then lowest mean FD of retained frames
# --------------------------------------------

assert SESSION_INPUT in ('combined', 'best_run'), SESSION_INPUT
print('XCP-D   :', XCPD_DIR, '| exists:', XCPD_DIR.exists())
print('fMRIPrep:', FMRIPREP_DIR, '| exists:', FMRIPREP_DIR.exists())
print('Outputs :', DFS, '|', FIG)
missing = [n for n, r in ROIS.items() if r['kind'] == 'nifti' and not Path(r['path']).exists()]
print('ROIs    :', list(ROIS), '| missing NIfTI:', missing or 'none')
print('Session input:', SESSION_INPUT)

## Updated Sheet
Reload the latest saved working tables from `dfs/` so later sections can be rerun without repeating the upstream pipeline. On a first run nothing is loaded.

In [ ]:
# ==========================================
# CELL 2: LOAD SAVED WORKING TABLES
# ==========================================
# Loads each saved working dataframe from DFS if present (written by CELLS 3-13).
# analysis_inputs (one file per session) is the input for connectivity analyses.

SAVED = ['runs', 'headers', 'motion_long', 'motion_qc', 'session_qc', 'roi_index', 'roi_qc',
         'roi_agree', 'run_qc', 'final_sample', 'analysis_inputs']
for name in SAVED:
    p = DFS / f'{name}.csv'
    if p.exists():
        globals()[name] = pd.read_csv(p, dtype={'subject': str, 'session': str, 'run': str, 'runs_used': str, 'runs_usable': str})
        print(f'loaded {name:15s} {globals()[name].shape}')
    else:
        print(f'not yet saved: {name}')

## 1. Inventory

In [ ]:
# ==========================================
# CELL 3: FILE INVENTORY
# ==========================================
# One row per subject/session/run with the path of every file used downstream
# (denoised + smoothed dtseries, DCAN HDF5, motion.tsv, atlas time series, fMRIPrep
# dtseries for tSNR). timepoint = session order within subject (1 = first session).

runs = ocs.find_runs(XCPD_DIR, task=TASK, atlases=tuple(ATLASES), fmriprep_dir=FMRIPREP_DIR)
file_cols = [c for c in runs.columns if c not in ('subject', 'session', 'task', 'run', 'timepoint')]
present = runs[file_cols].notna()
complete = present.all(axis=1)

print(f"{runs.subject.nunique()} subjects | {runs[['subject','session']].drop_duplicates().shape[0]} sessions | {len(runs)} runs")
print('\nFiles present (runs with file / total):')
print(present.sum().to_frame('n').assign(of=len(runs)).T.to_string())
incomplete = runs.loc[~complete, ['subject', 'session', 'run']].assign(
    missing=present[~complete].apply(lambda r: ', '.join(r.index[~r]), axis=1))
print(f'\nRuns with missing files: {len(incomplete)}')
if len(incomplete): print(incomplete.to_string(index=False))
print('\nRuns per subject x timepoint:')
print(runs.pivot_table(index='subject', columns='timepoint', values='run', aggfunc='count').fillna(0).astype(int).to_string())

runs.to_csv(DFS / 'runs.csv', index=False)

## 2. Header checks
XCP-D < 0.7 removes censored volumes from `desc-denoised_bold`, so the dtseries length should equal the DCAN *remaining* frame count at the FD threshold XCP-D used. Matching the two identifies that threshold per run. TR is read from the CIFTI series axis.

In [ ]:
# ==========================================
# CELL 4: HEADER CHECKS (TR, LENGTHS, APPLIED FD THRESHOLD)
# ==========================================
# Per run: TR, frames and grayordinates from the denoised dtseries header (no data
# loaded), uncensored length from motion.tsv and DCAN, and the FD thresholds whose DCAN
# remaining-frame count equals the dtseries length. Low-motion runs match many thresholds,
# so the threshold XCP-D applied (XCPD_FD_THR) is the one consistent with every run.

rows, matches = [], []
for r in runs.itertuples():
    row = dict(subject=r.subject, session=r.session, run=r.run)
    if pd.notna(r.dtseries):
        h = nib.load(r.dtseries).header
        row.update(tr=float(h.get_axis(0).step), n_frames=int(h.get_axis(0).size), n_grayords=int(h.get_axis(1).size))
    if pd.notna(r.motion_tsv):
        row['n_frames_motion_tsv'] = len(pd.read_csv(r.motion_tsv, sep='\t'))
    m = []
    if pd.notna(r.dcan_hdf5):
        dc = ocs.read_dcan_all(r.dcan_hdf5)
        row['n_frames_dcan_total'] = int(dc.frames_total.iloc[0])
        if 'n_frames' in row:
            m = ocs.matching_thresholds(dc, row['n_frames'])
            row['matching_fd_thr'] = f'{min(m):g}-{max(m):g}' if m else 'none'
    matches.append(m); rows.append(row)
headers = pd.DataFrame(rows)
XCPD_FD_THR = ocs.consensus_threshold(matches)

has_ts = headers['n_frames'].notna()
headers['flag_tr'] = has_ts & ~np.isclose(headers['tr'], EXPECTED_TR)
headers['flag_grayords'] = has_ts & (headers['n_grayords'] != EXPECTED_GRAYORDS)
headers['flag_length'] = headers['n_frames_motion_tsv'] != headers['n_frames_dcan_total']
headers['flag_thr'] = [bool(m) and XCPD_FD_THR not in m for m in matches]
headers['flag_no_dtseries'] = ~has_ts

print(f'FD threshold applied by XCP-D (consistent with all runs): {XCPD_FD_THR}')
print('TR values (s):', headers.loc[has_ts, 'tr'].round(3).value_counts().to_dict())
print('Uncensored frames (motion.tsv):', headers['n_frames_motion_tsv'].value_counts().to_dict())
flag_cols = [c for c in headers if c.startswith('flag_')]
for f in flag_cols:
    print(f'{f:18s}: {int(headers[f].sum())} runs')
flagged = headers[headers[flag_cols].any(axis=1)]
if len(flagged):
    shown = [c for c in headers if c not in flag_cols] + [c for c in flag_cols if flagged[c].any()]
    print('\nFlagged runs:\n' + flagged[shown].to_string(index=False))

headers.to_csv(DFS / 'headers.csv', index=False)

## 3. Motion QC (DCAN)
Retained frames/seconds and mean FD of retained frames come from the DCAN HDF5 at every FD threshold; uncensored FD statistics come from `motion.tsv`. Inclusion uses the threshold XCP-D applied (CELL 4) and the parameters in CELL 1.

In [ ]:
# ==========================================
# CELL 5: DCAN MOTION METRICS & RUN INCLUSION
# ==========================================
# motion_long: every run x every DCAN FD threshold (frames/seconds remaining, mean FD).
# motion_qc: one row per run at the XCP-D threshold + uncensored FD stats from motion.tsv,
# with run_ok = dtseries present & seconds_remaining >= MIN_RUN_SECONDS & mean FD <= MAX_MEAN_FD.

if XCPD_FD_THR is None:
    XCPD_FD_THR = 0.3
    warnings.warn('No FD threshold matched every run (CELL 4); falling back to 0.3 - check headers.csv')

ml = []
for r in runs.itertuples():
    if pd.notna(r.dcan_hdf5):
        ml.append(ocs.read_dcan_all(r.dcan_hdf5).assign(subject=r.subject, session=r.session, run=r.run))
motion_long = pd.concat(ml, ignore_index=True)
motion_long = motion_long[['subject', 'session', 'run'] + [c for c in motion_long if c not in ('subject', 'session', 'run')]]

at_thr = motion_long[np.isclose(motion_long.fd_threshold, XCPD_FD_THR)].drop(columns='fd_threshold')
fdstats = pd.DataFrame([dict(subject=r.subject, session=r.session, run=r.run, **ocs.fd_summary(r.motion_tsv, FD_REPORT))
                        for r in runs.itertuples() if pd.notna(r.motion_tsv)])
motion_qc = (runs[['subject', 'session', 'run', 'timepoint']]
             .assign(has_dtseries=runs.dtseries.notna())
             .merge(at_thr, on=['subject', 'session', 'run'], how='left')
             .merge(fdstats, on=['subject', 'session', 'run'], how='left'))
motion_qc['fd_threshold'] = XCPD_FD_THR
motion_qc['ok_seconds'] = motion_qc.seconds_remaining >= MIN_RUN_SECONDS
motion_qc['ok_mean_fd'] = motion_qc.mean_fd_remaining <= MAX_MEAN_FD
motion_qc['run_ok'] = motion_qc.has_dtseries & motion_qc.ok_seconds & motion_qc.ok_mean_fd

print(f'FD threshold {XCPD_FD_THR} mm | runs: {len(motion_qc)} | run_ok: {int(motion_qc.run_ok.sum())}')
print(f"  failing: no dtseries {int((~motion_qc.has_dtseries).sum())}, "
      f"< {MIN_RUN_SECONDS}s retained {int((~motion_qc.ok_seconds).sum())}, "
      f"mean FD > {MAX_MEAN_FD} {int((~motion_qc.ok_mean_fd).sum())}")
print('\nRetained minutes per run by timepoint:')
print((motion_qc.assign(min_remaining=motion_qc.seconds_remaining / 60)
       .groupby('timepoint')[['min_remaining', 'fraction_remaining', 'mean_fd_all', 'mean_fd_remaining']]
       .describe().T.loc[(slice(None), ['mean', 'std', 'min', 'max']), :]).round(3).to_string())

motion_long.to_csv(DFS / 'motion_long.csv', index=False)
motion_qc.to_csv(DFS / 'motion_qc.csv', index=False)

In [ ]:
# ==========================================
# CELL 6: SESSION & SUBJECT INCLUSION
# ==========================================
# session_qc: one row per subject/session; retained seconds summed over run_ok runs;
# session_ok if that total >= MIN_SESSION_SECONDS. Subjects with both timepoints ok
# form the paired (pre/post) sample.

session_qc = (motion_qc.assign(sec_ok=motion_qc.seconds_remaining.where(motion_qc.run_ok, 0))
              .groupby(['subject', 'session', 'timepoint'], as_index=False)
              .agg(n_runs=('run', 'size'), n_runs_ok=('run_ok', 'sum'),
                   seconds_ok=('sec_ok', 'sum'), mean_fd_all=('mean_fd_all', 'mean')))
session_qc['session_ok'] = session_qc.seconds_ok >= MIN_SESSION_SECONDS

by_sub = session_qc.pivot_table(index='subject', columns='timepoint', values='session_ok', aggfunc='first')
paired = by_sub.reindex(columns=[1, 2]).fillna(False).astype(bool).all(axis=1)
session_qc['paired_ok'] = session_qc.subject.map(paired)

print(f"Sessions: {len(session_qc)} | session_ok: {int(session_qc.session_ok.sum())} "
      f"(min {MIN_SESSION_SECONDS}s from runs passing CELL 5)")
print(session_qc.groupby('timepoint').agg(n_sessions=('session', 'size'), n_ok=('session_ok', 'sum'),
                                         median_min_ok=('seconds_ok', lambda s: s.median() / 60)).round(2).to_string())
print(f"\nSubjects: {session_qc.subject.nunique()} | with both timepoints usable (paired sample): {int(paired.sum())}")
excluded = session_qc[~session_qc.session_ok]
if len(excluded):
    print('\nExcluded sessions:\n' + excluded.assign(min_ok=excluded.seconds_ok / 60)
          [['subject', 'session', 'timepoint', 'n_runs', 'n_runs_ok', 'min_ok', 'mean_fd_all']].round(2).to_string(index=False))

session_qc.to_csv(DFS / 'session_qc.csv', index=False)

In [ ]:
# ==========================================
# CELL 7: MOTION QC FIGURES
# ==========================================
# A: retained minutes per run; B: fraction of frames retained vs FD threshold (median, IQR);
# C: mean FD (all frames) per session, paired across timepoints; D: usable minutes per session.
# Timepoint colors are categorical slots 1-2; failing runs/sessions are gray 'x' markers.

C_T = {1: '#2a78d6', 2: '#eb6834'}; C_X, C_GRID, C_AX = '#8a897f', '#e1e0d9', '#c3c2b7'
rng_j = np.random.default_rng(1)
def jit(n): return rng_j.uniform(-0.12, 0.12, n)
def style(ax):
    ax.grid(axis='y', color=C_GRID, lw=0.8); ax.set_axisbelow(True)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['left', 'bottom']: ax.spines[s].set_color(C_AX)

fig, axs = plt.subplots(2, 2, figsize=(12, 9), layout='constrained')
tps = sorted(motion_qc.timepoint.dropna().unique())

# A — retained minutes per run
ax = axs[0, 0]
for tp in tps:
    d = motion_qc[motion_qc.timepoint == tp]
    for ok, mk in [(True, 'o'), (False, 'x')]:
        dd = d[d.run_ok == ok]
        ax.scatter(tp + jit(len(dd)), dd.seconds_remaining / 60, s=40 if ok else 60, marker=mk,
                   color=C_T[tp] if ok else C_X, lw=2 if not ok else 0.5,
                   edgecolor='white' if ok else None, label=None)
ax.axhline(MIN_RUN_SECONDS / 60, color=C_AX, ls='--', lw=1.5)
ax.text(tps[-1] + 0.35, MIN_RUN_SECONDS / 60, f'min {MIN_RUN_SECONDS}s', va='bottom', ha='right', fontsize=9, color='#555')
ax.set_xticks(tps, [f'T{int(t)}' for t in tps]); ax.set_xlim(tps[0] - .5, tps[-1] + .5)
ax.set_ylim(bottom=0); ax.set_ylabel('minutes retained'); ax.set_title(f'A. Retained time per run (FD ≤ {XCPD_FD_THR} mm)', loc='left', fontweight='bold')
style(ax)

# B — retention curve across FD thresholds
ax = axs[0, 1]
q = motion_long.groupby('fd_threshold').fraction_remaining.quantile([.25, .5, .75]).unstack()
ax.fill_between(q.index, q[.25], q[.75], color=C_T[1], alpha=.2, lw=0, label='IQR across runs')
ax.plot(q.index, q[.5], color=C_T[1], lw=2, label='median')
ax.axvline(XCPD_FD_THR, color=C_AX, ls='--', lw=1.5)
ax.text(XCPD_FD_THR, 0.02, f' XCP-D {XCPD_FD_THR}', fontsize=9, color='#555')
ax.set_xlabel('FD threshold (mm)'); ax.set_ylabel('fraction of frames retained'); ax.set_ylim(0, 1.02)
ax.set_title('B. Retention vs FD threshold', loc='left', fontweight='bold'); ax.legend(frameon=False, loc='lower right')
style(ax)

# C — mean FD per session, paired lines
ax = axs[1, 0]
sq = session_qc.copy()
for s, d in sq.groupby('subject'):
    if len(d) > 1: ax.plot(d.timepoint, d.mean_fd_all, color=C_GRID, lw=1, zorder=1)
for tp in tps:
    d = sq[sq.timepoint == tp]
    ax.scatter(d.timepoint, d.mean_fd_all, s=40, color=C_T[tp], edgecolor='white', lw=.5, zorder=2)
ax.set_xticks(tps, [f'T{int(t)}' for t in tps]); ax.set_xlim(tps[0] - .5, tps[-1] + .5)
ax.set_ylabel('mean FD, all frames (mm)'); ax.set_title('C. Head motion per session (lines = same subject)', loc='left', fontweight='bold')
style(ax)

# D — usable minutes per session
ax = axs[1, 1]
for tp in tps:
    d = sq[sq.timepoint == tp]
    for ok, mk in [(True, 'o'), (False, 'x')]:
        dd = d[d.session_ok == ok]
        ax.scatter(tp + jit(len(dd)), dd.seconds_ok / 60, s=40 if ok else 60, marker=mk,
                   color=C_T[tp] if ok else C_X, lw=2 if not ok else 0.5, edgecolor='white' if ok else None)
ax.axhline(MIN_SESSION_SECONDS / 60, color=C_AX, ls='--', lw=1.5)
ax.text(tps[-1] + 0.35, MIN_SESSION_SECONDS / 60, f'min {MIN_SESSION_SECONDS}s', va='bottom', ha='right', fontsize=9, color='#555')
ax.set_xticks(tps, [f'T{int(t)}' for t in tps]); ax.set_xlim(tps[0] - .5, tps[-1] + .5)
ax.set_ylim(bottom=0); ax.set_ylabel('usable minutes (passing runs)'); ax.set_title('D. Usable time per session', loc='left', fontweight='bold')
style(ax)

from matplotlib.lines import Line2D
fig.legend(handles=[Line2D([], [], marker='o', ls='', color=C_T[1], label='T1 (first session)'),
                    Line2D([], [], marker='o', ls='', color=C_T[2], label='T2 (second session)'),
                    Line2D([], [], marker='x', ls='', color=C_X, mew=2, label='fails inclusion')],
           loc='outside lower center', ncol=3, frameon=False)
fig.suptitle(f'Motion QC — {len(motion_qc)} runs, {len(session_qc)} sessions, {session_qc.subject.nunique()} subjects', fontsize=14)
fig.savefig(FIG / 'motion_qc.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. ROI QC
Volumetric ROIs (NIfTI in MNI) are sampled at each CIFTI subcortical grayordinate (nearest voxel). XCP-D atlas parcels are read from the run's `*_timeseries.tsv`. Metrics are computed on the **unsmoothed** denoised dtseries (so neighbouring tissue is not blurred into the ROI); tSNR uses the fMRIPrep (un-denoised) dtseries.

**Caveat:** the CIT168 composite is in MNI152NLin2009cAsym, while CIFTI voxels are MNI152NLin6Asym, a ~1–2 mm mismatch. The Murty maps are on the matching FSL MNI grid.

In [ ]:
# ==========================================
# CELL 8: ROI -> GRAYORDINATE MAPPING
# ==========================================
# Maps each NIfTI ROI onto the 91k subcortical grayordinates of a reference dtseries
# (all runs share the same 91k brain model; CELL 4 checks grayordinate counts), split by
# hemisphere. ROIs with min_coverage use partial-volume sampling (see ocs.mask_from_nifti);
# missing NIfTI files are skipped. hemi M = midline x = 0 (bilateral ROI only).

ref_path = runs.dtseries.dropna().iloc[0]
BM = nib.load(ref_path).header.get_axis(1)
_, XYZ, STRUCT = ocs.volume_grayordinates(BM)
pos = {g: i for i, g in enumerate(np.where(BM.volume_mask)[0])}

ROI_IDX, rows = {}, []
for name, spec in ROIS.items():
    if spec['kind'] != 'nifti':
        continue
    if not Path(spec['path']).exists():
        warnings.warn(f'{name}: {spec["path"]} not found - ROI skipped'); continue
    idx = ocs.mask_from_nifti(spec['path'], BM, threshold=spec.get('threshold', 0.5), labels=spec.get('labels'),
                              min_coverage=spec.get('min_coverage'))
    ROI_IDX[name] = {'B': idx, **ocs.split_hemi(idx, BM)}   # B = bilateral incl. midline (x = 0)
    for g in idx:
        x, y, z = XYZ[pos[g]]
        rows.append(dict(roi=name, hemi='L' if x < 0 else 'R' if x > 0 else 'M', grayordinate=int(g),
                         x=x, y=y, z=z, structure=STRUCT[pos[g]].replace('CIFTI_STRUCTURE_', '')))
roi_index = pd.DataFrame(rows)

print(f'Reference brain model: {len(BM)} grayordinates ({int(BM.volume_mask.sum())} volumetric)')
summary = roi_index.groupby(['roi', 'hemi']).agg(n=('grayordinate', 'size'),
            x=('x', 'mean'), y=('y', 'mean'), z=('z', 'mean')).round(1)
print(summary.unstack('hemi').to_string())
print('\nGrayordinates by CIFTI structure:')
print(roi_index.groupby(['roi', 'structure']).size().unstack(fill_value=0).to_string())
empty = [n for n, d in ROI_IDX.items() if len(d['B']) == 0]
if empty: warnings.warn(f'ROIs with no grayordinates: {empty}')
small = [n for n, d in ROI_IDX.items() if 0 < len(d['B']) < 20]
if small: warnings.warn(f'ROIs with < 20 grayordinates (noisy seeds; interpret QC with care): {small}')

roi_index.to_csv(DFS / 'roi_index.csv', index=False)

In [ ]:
# ==========================================
# CELL 9: ROI QC PER RUN
# ==========================================
# For every run with a dtseries: loads the unsmoothed denoised data (+ fMRIPrep dtseries
# for tSNR), computes ocs.roi_qc for each NIfTI ROI x hemisphere (L, R, B = bilateral),
# and checks XCP-D atlas ROIs (missing/NaN columns). roi_agree holds the correlation
# between ROI-definition mean time series within each run (bilateral).

qc_rows, agree_rows = [], []
for k, r in enumerate(runs[runs.dtseries.notna()].itertuples(), 1):
    data, bm, tr = ocs.load_dtseries(r.dtseries)
    if len(bm) != len(BM):
        warnings.warn(f'{r.dtseries}: brain model differs from reference, skipped'); continue
    raw = ocs.load_dtseries(r.fmriprep_dtseries)[0] if pd.notna(r.fmriprep_dtseries) else None
    key = dict(subject=r.subject, session=r.session, run=r.run)
    ts = {}
    for name, halves in ROI_IDX.items():
        for h, idx in halves.items():
            qc_rows.append(dict(**key, roi=name, hemi=h, **ocs.roi_qc(data, idx, raw)))
        ts[name] = ocs.roi_timeseries(data, halves['B'])
    for name, spec in ROIS.items():
        if spec['kind'] != 'atlas':
            continue
        col = f"ts_{spec['atlas']}"
        tsv = pd.read_csv(getattr(r, col), sep='\t') if col in runs and pd.notna(getattr(r, col)) else None
        cols = [c for c in spec['columns'] if tsv is not None and c in tsv]
        row = dict(**key, roi=name, hemi='B', n_columns_found=len(cols), n_columns_expected=len(spec['columns']))
        if cols:
            v = tsv[cols]
            row['frac_valid'] = float(v.notna().all().mean())
            if len(v) == len(data):
                ts[name] = v.mean(axis=1).to_numpy()
        qc_rows.append(row)
    names = [n for n in ts if np.isfinite(ts[n]).all()]
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            agree_rows.append(dict(**key, roi_a=a, roi_b=b, r=float(np.corrcoef(ts[a], ts[b])[0, 1])))
    print(f'\r{k} runs processed', end='')
roi_qc = pd.DataFrame(qc_rows)
roi_agree = pd.DataFrame(agree_rows)

print('\n\nMedian ROI QC across runs (bilateral):')
cols = ['n_grayordinates', 'frac_valid', 'sd_ratio', 'homogeneity', 'pc1_var', 'r_global', 'tsnr_median']
print(roi_qc[roi_qc.hemi == 'B'].groupby('roi')[[c for c in cols if c in roi_qc]].median().round(3).to_string())
print('\nMedian correlation between ROI definitions (bilateral mean time series):')
print(roi_agree.groupby(['roi_a', 'roi_b']).r.median().round(3).to_string())

roi_qc.to_csv(DFS / 'roi_qc.csv', index=False)
roi_agree.to_csv(DFS / 'roi_agree.csv', index=False)

In [ ]:
# ==========================================
# CELL 10: ROI FLAGS & FINAL RUN TABLE
# ==========================================
# Flags FOCUS_ROI per run: coverage (frac_valid < MIN_ROI_COVERAGE in either hemisphere)
# and robust outliers (|median/MAD z| > ROI_Z_MAX across runs) in sd_ratio, homogeneity,
# r_global and tSNR (bilateral). run_qc = motion_qc + ROI flags; usable = run_ok & roi_ok.

MIN_ROI_COVERAGE = 0.90   # (CONFIRM)
ROI_Z_MAX = 3.0           # (CONFIRM)

def robust_z(s):
    mad = 1.4826 * np.nanmedian(np.abs(s - np.nanmedian(s)))
    return (s - np.nanmedian(s)) / mad if mad > 0 else s * 0

f = roi_qc[roi_qc.roi == FOCUS_ROI]
cov = f[f.hemi.isin(['L', 'R'])].pivot_table(index=['subject', 'session', 'run'], columns='hemi', values='frac_valid')
b = f[f.hemi == 'B'].set_index(['subject', 'session', 'run'])
flags = pd.DataFrame(index=b.index)
flags['roi_min_coverage'] = cov.min(axis=1)
flags['flag_roi_coverage'] = flags.roi_min_coverage < MIN_ROI_COVERAGE
for m in ['sd_ratio', 'homogeneity', 'r_global', 'tsnr_median']:
    if m in b:
        flags[f'z_{m}'] = robust_z(b[m])
        flags[f'flag_roi_{m}'] = flags[f'z_{m}'].abs() > ROI_Z_MAX
flag_cols = [c for c in flags if c.startswith('flag_roi')]
flags['roi_ok'] = ~flags[flag_cols].any(axis=1)

run_qc = motion_qc.merge(flags.reset_index(), on=['subject', 'session', 'run'], how='left')
run_qc['roi_ok'] = run_qc.roi_ok.fillna(False).astype(bool)
run_qc['usable'] = run_qc.run_ok & run_qc.roi_ok
run_qc['focus_roi'] = FOCUS_ROI

print(f'FOCUS_ROI = {FOCUS_ROI}')
for c in flag_cols:
    print(f'  {c:28s}: {int(flags[c].sum())} runs')
print(f"\nruns: {len(run_qc)} | motion ok: {int(run_qc.run_ok.sum())} | ROI ok: {int(run_qc.roi_ok.sum())} "
      f"| usable (both): {int(run_qc.usable.sum())}")
bad = run_qc[run_qc.run_ok & ~run_qc.roi_ok]
if len(bad):
    print('\nMotion-ok runs failing ROI QC:')
    show = ['subject', 'session', 'run', 'roi_min_coverage'] + [c for c in flag_cols if bad[c].any()]
    print(bad[show].round(3).to_string(index=False))

run_qc.to_csv(DFS / 'run_qc.csv', index=False)

In [ ]:
# ==========================================
# CELL 11: ROI QC FIGURES (FOCUS_ROI)
# ==========================================
# Per-run ROI metrics for FOCUS_ROI by timepoint (gray 'x' = run failing ROI or motion QC),
# ROI homogeneity vs motion, and agreement (median r of bilateral mean time series)
# between all ROI definitions.

from matplotlib.colors import LinearSegmentedColormap
d = run_qc.merge(roi_qc[(roi_qc.roi == FOCUS_ROI) & (roi_qc.hemi == 'B')], on=['subject', 'session', 'run'], how='inner')
d['fail'] = ~d.usable

def strip(ax, col, title, ylabel, ref=None, ref_lab=None):
    for tp in tps:
        dd = d[d.timepoint == tp]
        ok, bad = dd[~dd.fail], dd[dd.fail]
        ax.scatter(tp + jit(len(ok)), ok[col], s=40, color=C_T[tp], edgecolor='white', lw=.5)
        ax.scatter(tp + jit(len(bad)), bad[col], s=60, marker='x', color=C_X, lw=2)
    if ref is not None:
        ax.axhline(ref, color=C_AX, ls='--', lw=1.5)
        ax.text(tps[-1] + 0.4, ref, ref_lab, va='bottom', ha='right', fontsize=9, color='#555')
    ax.set_xticks(tps, [f'T{int(t)}' for t in tps]); ax.set_xlim(tps[0] - .5, tps[-1] + .5)
    ax.set_ylabel(ylabel); ax.set_title(title, loc='left', fontweight='bold'); style(ax)

fig, axs = plt.subplots(2, 3, figsize=(15, 9), layout='constrained')
strip(axs[0, 0], 'roi_min_coverage', 'A. Coverage (worse hemisphere)', 'fraction of ROI grayordinates valid',
      MIN_ROI_COVERAGE, f'min {MIN_ROI_COVERAGE}')
axs[0, 0].set_ylim(-0.02, 1.05)
if 'tsnr_median' in d and d.tsnr_median.notna().any():
    strip(axs[0, 1], 'tsnr_median', 'B. tSNR in ROI (fMRIPrep data)', 'median tSNR')
else:
    axs[0, 1].axis('off'); axs[0, 1].text(.5, .5, 'no fMRIPrep dtseries → no tSNR', ha='center')
strip(axs[0, 2], 'r_global', 'C. ROI ↔ global signal', 'r (ROI mean, global mean)')

ax = axs[1, 0]
ok = d[~d.fail]; bad = d[d.fail]
for tp in tps:
    o = ok[ok.timepoint == tp]
    ax.scatter(o.mean_fd_remaining, o.homogeneity, s=40, color=C_T[tp], edgecolor='white', lw=.5)
ax.scatter(bad.mean_fd_remaining, bad.homogeneity, s=60, marker='x', color=C_X, lw=2)
ax.set_xlabel('mean FD of retained frames (mm)'); ax.set_ylabel('ROI homogeneity (mean pairwise r)')
ax.set_title('D. ROI coherence vs residual motion', loc='left', fontweight='bold'); style(ax)

strip(axs[1, 1], 'sd_ratio', 'E. ROI signal amplitude', 'ROI SD / whole-brain SD')

ax = axs[1, 2]
names = list(ROIS)
M = pd.DataFrame(np.eye(len(names)), index=names, columns=names)
for (a, b_), v in roi_agree.groupby(['roi_a', 'roi_b']).r.median().items():
    M.loc[a, b_] = M.loc[b_, a] = v
cm_seq = LinearSegmentedColormap.from_list('seq', ['#cde2fb', '#2a78d6', '#0d366b'])
im = ax.imshow(M.values, cmap=cm_seq, vmin=0, vmax=1)
for i in range(len(names)):
    for j in range(len(names)):
        v = M.values[i, j]
        if np.isfinite(v): ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9,
                                    color='white' if v > 0.6 else '#0b0b0b')
short = [n.replace('_', '\n', 1) for n in names]
ax.set_xticks(range(len(names)), short, fontsize=8, rotation=45, ha='right', rotation_mode='anchor'); ax.set_yticks(range(len(names)), short, fontsize=8)
ax.set_title('F. Agreement between ROI definitions', loc='left', fontweight='bold')
fig.colorbar(im, ax=ax, shrink=.7, label='median r across runs')

fig.legend(handles=[Line2D([], [], marker='o', ls='', color=C_T[1], label='T1'),
                    Line2D([], [], marker='o', ls='', color=C_T[2], label='T2'),
                    Line2D([], [], marker='x', ls='', color=C_X, mew=2, label='not usable (ROI or motion QC)')],
           loc='outside lower center', ncol=3, frameon=False)
fig.suptitle(f'ROI QC — {FOCUS_ROI} (bilateral, unsmoothed denoised data)', fontsize=14)
fig.savefig(FIG / f'roi_qc_{FOCUS_ROI}.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Final analysis sample

In [ ]:
# ==========================================
# CELL 12: FINAL ANALYSIS SAMPLE
# ==========================================
# Session inclusion recomputed from runs passing BOTH motion and FOCUS_ROI QC (usable).
# final_sample: one row per subject/session with usable runs, usable seconds and
# whether the subject has both timepoints usable (paired pre/post sample).

final_sample = (run_qc.assign(sec_usable=run_qc.seconds_remaining.where(run_qc.usable, 0),
                              run_usable=run_qc.run.where(run_qc.usable))
                .groupby(['subject', 'session', 'timepoint'], as_index=False)
                .agg(n_runs=('run', 'size'), n_runs_usable=('usable', 'sum'), seconds_usable=('sec_usable', 'sum'),
                     runs_usable=('run_usable', lambda s: ','.join(s.dropna()))))
final_sample['session_usable'] = final_sample.seconds_usable >= MIN_SESSION_SECONDS
both = (final_sample.pivot_table(index='subject', columns='timepoint', values='session_usable', aggfunc='first')
        .reindex(columns=[1, 2]).fillna(False).astype(bool).all(axis=1))
final_sample['paired_usable'] = final_sample.subject.map(both)
final_sample['focus_roi'] = FOCUS_ROI

print(f'Criteria: FD <= {XCPD_FD_THR} mm (XCP-D), run >= {MIN_RUN_SECONDS}s & mean FD <= {MAX_MEAN_FD}, '
      f'{FOCUS_ROI} coverage >= {MIN_ROI_COVERAGE} & no |z| > {ROI_Z_MAX}, session >= {MIN_SESSION_SECONDS}s')
print(f"\nSessions usable: {int(final_sample.session_usable.sum())} / {len(final_sample)}")
print(final_sample.groupby('timepoint').agg(n=('session', 'size'), usable=('session_usable', 'sum'),
      median_usable_min=('seconds_usable', lambda s: s.median() / 60)).round(2).to_string())
print(f"\nSubjects with both timepoints usable (paired sample): {int(both.sum())} / {final_sample.subject.nunique()}")
print('\nAttrition (runs): total {} -> motion ok {} -> + ROI ok {}'.format(
    len(run_qc), int(run_qc.run_ok.sum()), int(run_qc.usable.sum())))

final_sample.to_csv(DFS / 'final_sample.csv', index=False)

## 6. Analysis input per session
One dtseries per session for connectivity. `SESSION_INPUT` (CELL 1) chooses:
- **`combined`** (default): the run-concatenated session file from `scripts/combine_runs.py`, used only when the runs it was built from are exactly the session's usable runs (CELL 10). Otherwise the session falls back to its best usable run and `reason` says why: one usable run, no combined file yet, or the combined file contains a run that failed QC.
- **`best_run`**: the single usable run ranked best by `BEST_RUN_RANK`, by default the most frames retained after censoring, then the lowest mean FD of the retained frames.

A session is included (`input_ok`) if its chosen input has at least `MIN_SESSION_SECONDS` of retained data.

In [ ]:
# ==========================================
# CELL 13: ANALYSIS INPUT PER SESSION (COMBINED / BEST RUN)
# ==========================================
# Picks one dtseries per session per SESSION_INPUT. Combined files come from
# ocs.find_combined (runs read from the sidecar); the best run is ranked by BEST_RUN_RANK among
# usable runs (run_qc). seconds/mean FD describe the retained frames of the chosen input.

comb = ocs.find_combined(XCPD_DIR, task=TASK)
rq = run_qc.merge(runs[['subject', 'session', 'run', 'dtseries', 'dtseries_smoothed']],
                  on=['subject', 'session', 'run'], how='left')
rank_cols, rank_asc = [c for c, _ in BEST_RUN_RANK], [a for _, a in BEST_RUN_RANK]

rows = []
for (sub, ses, tp), d in rq.groupby(['subject', 'session', 'timepoint']):
    u = d[d.usable].sort_values(rank_cols, ascending=rank_asc)
    usable = sorted(u.run)
    c = comb[(comb.subject == sub) & (comb.session == ses)]
    c_runs = sorted(r for r in c.runs.iloc[0].split(',') if r) if len(c) else []
    row = dict(subject=sub, session=ses, timepoint=tp, mode=SESSION_INPUT,
               runs_usable=','.join(usable), n_runs_usable=len(usable))
    if not usable:
        row.update(input='none', reason='no usable run')
    elif SESSION_INPUT == 'combined' and len(usable) >= 2 and len(c) and c.kind.iloc[0] == 'combined' and c_runs == usable:
        s = u.seconds_remaining
        row.update(input='combined', runs_used=','.join(usable), reason='combined usable runs',
                   dtseries=c.dtseries.iloc[0], dtseries_smoothed=c.dtseries_smoothed.iloc[0],
                   seconds=s.sum(), mean_fd_remaining=float((u.mean_fd_remaining * s).sum() / s.sum()))
    else:
        b = u.iloc[0]
        if SESSION_INPUT == 'best_run':
            reason = f'best of {len(usable)} usable run(s)'
        elif len(usable) == 1:
            reason = 'only 1 usable run'
        elif not len(c):
            reason = 'no combined file (run scripts/combine_runs.py)'
        elif c.kind.iloc[0] != 'combined':
            reason = f'combined file not verifiable ({c.kind.iloc[0]})'
        else:
            reason = f'combined file has runs {",".join(c_runs)}, usable runs are {",".join(usable)}'
        row.update(input='run', runs_used=b.run, reason=reason, dtseries=b.dtseries,
                   dtseries_smoothed=b.dtseries_smoothed, seconds=b.seconds_remaining,
                   mean_fd_remaining=b.mean_fd_remaining)
    rows.append(row)
analysis_inputs = pd.DataFrame(rows)
analysis_inputs['input_ok'] = analysis_inputs.seconds.fillna(0) >= MIN_SESSION_SECONDS
both = (analysis_inputs.pivot_table(index='subject', columns='timepoint', values='input_ok', aggfunc='first')
        .reindex(columns=[1, 2]).fillna(False).astype(bool).all(axis=1))
analysis_inputs['paired_ok'] = analysis_inputs.subject.map(both)

print(f'SESSION_INPUT = {SESSION_INPUT} | combined files found: {len(comb)} '
      f'({comb.kind.value_counts().to_dict()})')
print('\nInput used per session:')
print(analysis_inputs.groupby(['input', 'reason']).size().to_frame('sessions').to_string())
print(f"\nSessions with input >= {MIN_SESSION_SECONDS}s: {int(analysis_inputs.input_ok.sum())} / {len(analysis_inputs)}")
print(analysis_inputs.groupby('timepoint').agg(n=('session', 'size'), ok=('input_ok', 'sum'),
      median_min=('seconds', lambda s: s.median() / 60), median_fd=('mean_fd_remaining', 'median')).round(3).to_string())
print(f"Subjects with both timepoints ok (paired sample): {int(both.sum())} / {analysis_inputs.subject.nunique()}")

analysis_inputs.to_csv(DFS / 'analysis_inputs.csv', index=False)